# LanceDB fundamentals

In [20]:
import lancedb 
from constants import KNOWLEDGE_BASE_PATH

vector_db = lancedb.connect(KNOWLEDGE_BASE_PATH)
vector_db

LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base')

In [21]:
import json

with open("animals_text_embeddings.json") as file:
    data = json.load(file)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [22]:
vector_db.create_table("animals_text", exist_ok=True, data=data)
vector_db["animals_text"]

LanceTable(name='animals_text', version=3, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base'))

In [23]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [24]:
vector_db["animals_text"].head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

## Add more data

In [25]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

vector_db["animals_text"].add(more_data)

AddResult(version=4)

In [26]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table

In [27]:
from lancedb.pydantic import LanceModel

class Article(LanceModel):
    title: str 
    content: str 

vector_db.create_table("articles", schema=Article, exist_ok=True)


LanceTable(name='articles', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base'))

In [28]:
vector_db.list_tables()

ListTablesResponse(tables=['animals_text', 'articles', 'jokes'], page_token=None)

In [29]:
articles_data = [
    {
        "title": "Fiskar",
        "content": """Fiskar (Pisces) är en grupp vattenlevande ryggradsdjur med fenor, som indelas i benfiskar, broskfiskar och käklösa fiskar. De flesta arter andas med gälar och är växelvarma. Undantaget är lungfiskar. Eftersom landryggradsdjuren släktskapsmässigt är en typ av kvastfeniga fiskar men inte klassificeras som fiskar är begreppet "fiskar" parafyletiskt. Vetenskapen om fiskar kallas iktyologi.""",
    },
    {
        "title": "Krabba",
        "content": """Krabbor (Brachyura) är en delordning av vattenlevande kräftdjur. Vissa arter fiskas som mat, ofta ansedda som delikatesser. Krabbor har tio ben varav det första paret bär gripklor.""",
    },
]

vector_db["articles"].add(articles_data)

AddResult(version=2)

In [30]:
vector_db["articles"].to_pandas()

,title,content
0,Fiskar,Fiskar (Pisces) är en grupp vattenlevande rygg...
1,Krabba,Krabbor (Brachyura) är en delordning av vatten...


## Drop a table

In [31]:
vector_db.drop_table("articles")

In [32]:
vector_db.list_tables() # .table_names() for intel macs

ListTablesResponse(tables=['animals_text', 'jokes'], page_token=None)

## Vector search in LanceDB

Flow here (common approach in many vector databases): 
1. use embedding model to create vector embeddings for each document
2. use same embedding model to create vector embedding of the query
3. send in the query_vector to search for closest documents
   
If we want to do RAG -> send in closest document to LLM to use as context

In [33]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [34]:
query_vector = [0.5, 0.2, 0.9]
df = vector_db["animals_text"].search(query_vector).limit(3).to_pandas()
df

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354


## Embeddings API in LanceDB

- makes life simpler
- automatically generate embeddings

embedding models can be found here for cohere
- [embedding models](https://docs.cohere.com/docs/cohere-embed)

In [35]:
from lancedb.embeddings import get_registry
from dotenv import load_dotenv

load_dotenv()

embedding_model = get_registry().get("cohere").create(name = "embed-multilingual-v3.0")

embedding_model.ndims()

1024

In [36]:
from lancedb.pydantic import Vector

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField() # type:ignore

vector_db.create_table("jokes", schema=JokeModel, exist_ok=True)
vector_db["jokes"]



LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/github/llmops_kokchun_giang_mlo25/09_lancedb/knowledge_base'))

In [37]:
import pandas as pd 
with open("jokes.json", encoding="utf8") as file:
    jokes_data = json.load(file)

df_jokes = pd.DataFrame(jokes_data)
df_jokes.head()


,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [38]:

vector_db["jokes"].add(df_jokes)

AddResult(version=2)

In [41]:
df_jokes_embeddings = vector_db["jokes"].to_pandas()
df_jokes_embeddings.head()

,joke,embedding
0,Parallel lines have so much in common—it’s sad...,"[-0.0032196045, 0.0011167526, -0.014953613, 0...."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.012634277, 0.005268097, -0.09136963, 0.0809..."
2,What do you call a snake that runs your script...,"[0.019165039, 0.044677734, -0.021530151, 0.028..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013122559, 0.002374649, -0.05706787, 0.0608..."


In [45]:
df_jokes_embeddings["embedding"][10] # note dimensions 1024

array([ 0.01750183,  0.02137756, -0.01231384, ...,  0.05526733,
        0.04537964,  0.01043701], shape=(1024,), dtype=float32)

In [48]:
vector_db["jokes"].search("chemstry related joke").to_pandas()

,joke,embedding,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.03274536, 0.018218994, 0.039276123, 0.0012...",0.850965
1,Chemists have all the solutions… mostly in bea...,"[0.0013589859, 0.03086853, -0.0012874603, 0.02...",0.995148
2,Why did the chemist ground his kids? Because t...,"[-0.008056641, 0.031555176, -0.019454956, 0.01...",1.026592
3,Oxygen and Magnesium got together—OMg!,"[-0.04119873, -0.004600525, 0.007701874, 0.006...",1.040359
4,The C# compiler walked into a bar. The bartend...,"[0.0038490295, 0.006790161, -0.0769043, 0.0265...",1.104045
5,Why’s 6 afraid of 7? Because 7 8 9.,"[0.005886078, 0.020553589, -0.0031318665, 0.01...",1.110861
6,Never trust an atom—they make up everything.,"[-0.008956909, 0.003440857, -0.020614624, 0.04...",1.144259
7,"Gold walks into a bar. The bartender says, “Au...","[0.019866943, 0.05999756, 0.012702942, 0.02343...",1.172260
8,I put “root beer” in a square glass… now it’s ...,"[0.008476257, 0.026138306, 0.046020508, 0.0015...",1.202099
9,What do you call a snake that runs your script...,"[0.019165039, 0.044677734, -0.021530151, 0.028...",1.202766
